In [1]:
# Day 4: Load Master Dataset to PostgreSQL Warehouse
# =====================================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv


In [2]:
load_dotenv('../.env')
engine = create_engine(os.getenv('DATABASE_URL'))
print("✅ Connected to Supabase PostgreSQL")

# Load master dataset
master_df = pd.read_csv('../data/processed/master_events.csv')
master_df['timestamp'] = pd.to_datetime(master_df['timestamp'])
print(f"✅ Loaded master dataset: {len(master_df):,} rows")

✅ Connected to Supabase PostgreSQL
✅ Loaded master dataset: 134,057 rows


In [3]:
# ===== BUILD DIMENSION TABLES =====

# dim_users
dim_users = master_df.groupby('user_id').agg(
    first_seen=('timestamp', 'min'),
    last_seen=('timestamp', 'max'),
    total_events=('event_type', 'count')
).reset_index()
print(f"dim_users: {len(dim_users):,} rows")

# dim_products
dim_products = master_df[master_df['product_id'].notna()][['product_id']].drop_duplicates().reset_index(drop=True)
dim_products['product_key'] = range(1, len(dim_products) + 1)
print(f"dim_products: {len(dim_products):,} rows")

# dim_dates
date_range = pd.date_range(master_df['timestamp'].min().date(), master_df['timestamp'].max().date(), freq='D')
dim_dates = pd.DataFrame({'date': date_range})
dim_dates['year'] = dim_dates['date'].dt.year
dim_dates['month'] = dim_dates['date'].dt.month
dim_dates['day'] = dim_dates['date'].dt.day
dim_dates['day_of_week'] = dim_dates['date'].dt.day_name()
dim_dates['quarter'] = dim_dates['date'].dt.quarter
print(f"dim_dates: {len(dim_dates):,} rows")

# fact_events (the main fact table)
fact_events = master_df[['user_id', 'session_id', 'timestamp', 'event_type', 'product_id', 'amount']].copy()
fact_events['event_id'] = range(1, len(fact_events) + 1)
print(f"fact_events: {len(fact_events):,} rows")

dim_users: 5,000 rows
dim_products: 8,982 rows
dim_dates: 365 rows
fact_events: 134,057 rows


In [4]:
# ===== LOAD TO POSTGRESQL =====
print("\nLoading tables to Supabase (this may take 2-3 minutes)...")

dim_users.to_sql('dim_users', engine, if_exists='replace', index=False, chunksize=1000)
print("✅ dim_users loaded")

dim_products.to_sql('dim_products', engine, if_exists='replace', index=False, chunksize=1000)
print("✅ dim_products loaded")

dim_dates.to_sql('dim_dates', engine, if_exists='replace', index=False, chunksize=1000)
print("✅ dim_dates loaded")

fact_events.to_sql('fact_events', engine, if_exists='replace', index=False, chunksize=5000)
print("✅ fact_events loaded")

# ===== VERIFY =====
print("\n" + "="*50)
print("VERIFICATION - Row counts in PostgreSQL:")
print("="*50)
with engine.connect() as conn:
    for table in ['dim_users', 'dim_products', 'dim_dates', 'fact_events']:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.fetchone()[0]
        print(f"  {table:15s}: {count:,} rows")

print("\n🎉 Day 4 complete! Data warehouse is live!")


Loading tables to Supabase (this may take 2-3 minutes)...
✅ dim_users loaded
✅ dim_products loaded
✅ dim_dates loaded
✅ fact_events loaded

VERIFICATION - Row counts in PostgreSQL:
  dim_users      : 5,000 rows
  dim_products   : 8,982 rows
  dim_dates      : 365 rows
  fact_events    : 134,057 rows

🎉 Day 4 complete! Data warehouse is live!
